# Whisper Streaming Server on Colab GPU
Run all cells top to bottom. Cell 6 prints the public host:port to put in your local `mic_client.py`.

> **Before running:** Runtime -> Change runtime type -> T4 GPU

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q faster-whisper librosa soundfile pyngrok

In [ ]:
!git clone --depth 1 https://github.com/ufal/whisper_streaming.git
print('Cloned!')

In [ ]:
# Patch whisper_online.py to use CUDA float16
import re
path = 'whisper_streaming/whisper_online.py'
with open(path, 'r') as f:
    content = f.read()
content = content.replace(
    'device="cpu", compute_type="int8"',
    'device="cuda", compute_type="float16"'
)
with open(path, 'w') as f:
    f.write(content)
print('Patched to CUDA float16 OK')

In [ ]:
# Get your FREE ngrok token at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = 'PASTE_YOUR_NGROK_TOKEN_HERE'

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_AUTH_TOKEN
print('ngrok token set OK')

In [ ]:
import subprocess
from pyngrok import ngrok

PORT = 43007

# Start whisper streaming server in background
server = subprocess.Popen(
    ['python3', 'whisper_streaming/whisper_online_server.py',
     '--language', 'en', '--model', 'base.en',
     '--host', '0.0.0.0', '--port', str(PORT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait until model is loaded
print('Loading Whisper base.en on GPU... (30-60 seconds)')
for line in server.stdout:
    print(line, end='')
    if 'Listening on' in line:
        break

# Create public TCP tunnel via ngrok
tunnel = ngrok.connect(PORT, 'tcp')
public_url = tunnel.public_url  # e.g. tcp://0.tcp.ngrok.io:12345
host, port = public_url.replace('tcp://', '').split(':')

print()
print('=' * 55)
print('  WHISPER SERVER IS LIVE!')
print(f'  Public URL : {public_url}')
print()
print('  In your LOCAL mic_client.py, update:')
print(f'    HOST = "{host}"')
print(f'    PORT = {port}')
print('=' * 55)

In [ ]:
# Keep this running to see live transcription logs
try:
    for line in server.stdout:
        if line.strip() and 'DEBUG' not in line:
            print(line, end='', flush=True)
except KeyboardInterrupt:
    print('Stopped log stream.')